# Paper-headline R² JSONs — §8 / §9 fit-script orchestration

## What this notebook produces

Runs the four `paper_tmlr_1` §8 / §9 fit scripts (`shared_potential_fit.py`, `splm_oracle_fit.py`, `sharedV_capacity_sweep.py`, `token_direction_fit.py`) on the three reference architectures of the three-way separator and aggregates the outputs into the five paper-headline JSONs named in `results/README.md`:

| Output JSON | Paper § | Source script |
|---|---|---|
| `R2_three_way_separator.json` | §8 | aggregator over three `shared_potential_fit.py` outputs |
| `R2_oracle.json` | §9.1 | `splm_oracle_fit.py` on the SPLM leak-free checkpoint |
| `R2_capacity_sweep.json` | §9.2 | `sharedV_capacity_sweep.py` on GPT-2 trajectories |
| `R2_token_direction.json` (× 3, one per architecture) | §9.3 | `token_direction_fit.py` on each trajectory |

Each JSON carries an `_provenance` block (git SHA, config hash, checkpoint SHA256 where applicable, random seed, timestamp, host) so any cited number can be traced back to the exact script revision, configuration, and checkpoint that produced it.

## Architectures covered

1. **Pretrained GPT-2 small** — public HF checkpoint, extracted on the fly.
2. **Matched-attention 8M baseline** — local `.pt` checkpoint, uploaded to GDrive once at the path in `MATCHED_CKPT_GDRIVE_REL`.
3. **SPLM positive control (leak-free, γ* = 0.10)** — local `.pt` checkpoint, uploaded to GDrive once at the path in `SPLM_CKPT_GDRIVE_REL`.

If a local checkpoint is absent the corresponding architecture is skipped and the downstream aggregation step prints a clear warning instead of failing.

## Wall-clock budget (H100)

| Stage | Architecture(s) | Wall-clock |
|---|---|---|
| Extract GPT-2 small trajectories | gpt2 | ~3 min |
| Extract matched-attention trajectories | local ckpt | ~2 min |
| Extract SPLM trajectories | local ckpt | ~2 min |
| `shared_potential_fit.py` × 3 (4000 steps each) | per-arch | ~5 min each (~15 min total) |
| `splm_oracle_fit.py` | SPLM only | < 1 min (closed-form LS) |
| `sharedV_capacity_sweep.py` (6 cells) | GPT-2 only | ~30 min |
| `token_direction_fit.py` × 3 | per-arch | ~5 min each (~15 min total) |
| Aggregate §8 three-way separator | --- | < 30 sec |

**Total: ≲ 1 h with all three architectures.**

## Getting the two local checkpoints (pick one option)

The matched-attention and SPLM arms need two locally-trained checkpoints. You have two equivalent ways to provide them:

**Option A — recreate them in-session (no upload).** Flip `TRAIN_CHECKPOINTS = True` in the **§2b** cell below. It runs [`make_checkpoints.py`](../make_checkpoints.py), which trains both architectures and writes them under `<repo>/checkpoints/` with the exact filenames the extraction stages expect, then (on Colab) persists them to `/MyDrive/paper_tmlr_1_checkpoints/` for reuse. Full paper-grade training is ~6 h on one GPU; a `'smoke'` mode runs in minutes for a non-paper-grade pipeline check.

**Option B — upload pre-trained checkpoints once.** Place the two checkpoints under `/MyDrive/paper_tmlr_1_checkpoints/`:

1. SPLM leak-free positive control — produced by `make_checkpoints.py --which splm` (equivalently `ln_damping_sweep/train_splm_em_ln.py --mode shakespeare --seed 0 --fixed-gamma 0.10 --tag-suffix gamma0p10_seed0`; see `checkpoints/README.md`) → upload as `paper_tmlr_1_checkpoints/splm_em_ln_leakfree_gamma0p10_seed0_ckpt_latest.pt`.
2. MatchedGPT 8M baseline — produced by `make_checkpoints.py --which matched` (equivalently `train_matched.py --mode shakespeare --seed 0`; see `checkpoints/README.md`) → upload as `paper_tmlr_1_checkpoints/matched_baseline_shakespeare_seed0_ckpt_latest.pt`.

The notebook reads them directly from GDrive at run time (no copying). If a checkpoint is absent and `TRAIN_CHECKPOINTS = False`, the corresponding architecture is skipped with a clear warning instead of failing.

## 0. Configuration + Colab bootstrap

**Configurable knobs** (edit the next code cell):
- `RUN_GPT2` / `RUN_MATCHED` / `RUN_SPLM` — architecture toggles; the local-checkpoint ones auto-skip if the GDrive `.pt` is absent.
- `MATCHED_CKPT_GDRIVE_REL` / `SPLM_CKPT_GDRIVE_REL` — relative paths under `/MyDrive/`.
- `SHARED_V_*` / `TOKEN_DIR_*` — fit hyperparameters (defaults match the paper).

**Colab bootstrap (automatic).** On Colab, mounts GDrive, shallow-clones `dimitarpg13/paper_tmlr_1`, routes all outputs to `/MyDrive/paper_tmlr_1_r2_jsons/`. Off Colab, walks up from the CWD to find the standalone repo root.

In [ ]:
# ===== Architecture toggles =====
RUN_GPT2    = True   # always available (public HF)
RUN_MATCHED = True   # auto-skipped if MATCHED_CKPT_GDRIVE_REL absent
RUN_SPLM    = True   # auto-skipped if SPLM_CKPT_GDRIVE_REL absent

# ===== Local-checkpoint paths (relative to /MyDrive/), ignored if absent =====
MATCHED_CKPT_GDRIVE_REL = 'paper_tmlr_1_checkpoints/matched_baseline_shakespeare_seed0_ckpt_latest.pt'
SPLM_CKPT_GDRIVE_REL    = 'paper_tmlr_1_checkpoints/splm_em_ln_leakfree_gamma0p10_seed0_ckpt_latest.pt'

# ===== Trajectory-extraction config (matches paper §8 / §9 protocol) =====
N_TEST_PER_DOMAIN = 2
MAX_LEN           = 64
SEED              = 0

# ===== Fit hyperparameters (match the defaults of each script) =====
SHARED_V_HIDDEN  = 256
SHARED_V_DEPTH   = 2
SHARED_V_STEPS   = 4000
SHARED_V_BATCH   = 2048
SHARED_V_LR      = 3e-3

TOKEN_DIR_HIDDEN = 256
TOKEN_DIR_DEPTH  = 2
TOKEN_DIR_STEPS  = 4000
TOKEN_DIR_BATCH  = 2048
TOKEN_DIR_LR     = 3e-3
TOKEN_DIR_PCA_K  = 16
TOKEN_DIR_RIDGE  = 1e-3
TOKEN_DIR_T_SKIP = 2

# ===== Source-of-truth repo + GDrive output dir =====
REPO_URL          = 'https://github.com/dimitarpg13/paper_tmlr_1.git'
REPO_BRANCH       = 'main'
COLAB_REPO_PATH   = '/content/paper_tmlr_1'
GDRIVE_OUT_REL    = 'paper_tmlr_1_r2_jsons'

import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}', flush=True)


def _sh(cmd):
    print(f'$ {cmd}', flush=True)
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')


if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_OUT = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'GDrive output root = {GDRIVE_OUT}', flush=True)

    REPO_ROOT = Path(COLAB_REPO_PATH)
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}')
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: could not refresh repo ({e}); using existing checkout.')

    RESULTS_ROOT = GDRIVE_OUT
    MATCHED_CKPT = Path('/content/drive/MyDrive') / MATCHED_CKPT_GDRIVE_REL
    SPLM_CKPT    = Path('/content/drive/MyDrive') / SPLM_CKPT_GDRIVE_REL

    _sh('pip install -q transformers huggingface_hub tokenizers')
else:
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'notebooks').exists():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / 'notebooks').exists():
        raise RuntimeError(
            'Could not locate the paper_tmlr_1 repo root from the notebook CWD. '
            'cd into the standalone paper_tmlr_1 repo before launching the notebook.'
        )
    RESULTS_ROOT = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scripts' / 'results' / 'r2_jsons'
    MATCHED_CKPT = REPO_ROOT / 'checkpoints' / Path(MATCHED_CKPT_GDRIVE_REL).name
    SPLM_CKPT    = REPO_ROOT / 'checkpoints' / Path(SPLM_CKPT_GDRIVE_REL).name

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

CONS_DIR    = REPO_ROOT / 'notebooks' / 'conservative_arch'
SCRIPTS_DIR = CONS_DIR / 'scripts'
(CONS_DIR / 'results').mkdir(parents=True, exist_ok=True)

print('', flush=True)
print(f'REPO_ROOT     = {REPO_ROOT}', flush=True)
print(f'CONS_DIR      = {CONS_DIR}', flush=True)
print(f'SCRIPTS_DIR   = {SCRIPTS_DIR}', flush=True)
print(f'RESULTS_ROOT  = {RESULTS_ROOT}', flush=True)
print(f'MATCHED_CKPT  = {MATCHED_CKPT}  (exists: {MATCHED_CKPT.exists()})', flush=True)
print(f'SPLM_CKPT     = {SPLM_CKPT}  (exists: {SPLM_CKPT.exists()})', flush=True)

if RUN_MATCHED and not MATCHED_CKPT.exists():
    print(f'WARNING: MATCHED_CKPT absent — matched-attention runs will be skipped.', flush=True)
    RUN_MATCHED = False
if RUN_SPLM and not SPLM_CKPT.exists():
    print(f'WARNING: SPLM_CKPT absent — SPLM runs (and the §8 three-way aggregator) will be skipped.', flush=True)
    RUN_SPLM = False

## 1. Disable TF32, set seeds, pick device

The fit scripts default to FP32. TF32 is disabled to keep R² numbers bit-for-bit comparable across reruns.

In [ ]:
import torch
import numpy as np

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.manual_seed_all(SEED)
    cap = torch.cuda.get_device_capability()
    print(f'CUDA: {torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}  '
          f'mem={torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB', flush=True)
    if cap[0] < 8:
        print(f'  WARNING: compute capability < 8.0 ({cap}); designed for A100 / H100.', flush=True)
elif torch.backends.mps.is_available():
    device = 'mps'
    print('MPS device — TF32 toggles are CUDA-only.', flush=True)
else:
    device = 'cpu'
    print('CPU only — expect substantially slower fits.', flush=True)
print(f'\ndevice = {device}', flush=True)

## 2. Helper: stream a subprocess's stdout live to the notebook

In [ ]:
import time, subprocess, shlex

def stream_subprocess(cmd, cwd=None, env=None, label=None):
    if isinstance(cmd, str):
        cmd_list = shlex.split(cmd)
    else:
        cmd_list = [str(c) for c in cmd]
    if label is None:
        label = Path(cmd_list[0]).name
    cmd_repr = ' '.join(shlex.quote(c) for c in cmd_list)
    print(f'\n[{label}] $ {cmd_repr}', flush=True)
    if cwd is not None:
        print(f'[{label}]   cwd = {cwd}', flush=True)
    t0 = time.time()
    p = subprocess.Popen(
        cmd_list, cwd=str(cwd) if cwd is not None else None, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, universal_newlines=True, text=True,
    )
    assert p.stdout is not None
    for line in p.stdout:
        sys.stdout.write(f'[{label}] {line}')
        sys.stdout.flush()
    rc = p.wait()
    elapsed = time.time() - t0
    print(f'[{label}] exit={rc}  elapsed={elapsed:.1f}s', flush=True)
    if rc != 0:
        raise RuntimeError(f'[{label}] command failed with exit code {rc}')
    return rc, elapsed

## 2b. (Optional) — (re)create the SPLM + matched checkpoints from scratch

The headline §8/§9 JSONs need two locally-trained checkpoints (the SPLM
leak-free positive control and the matched-attention 8M baseline). If you have
not uploaded them to `/MyDrive/paper_tmlr_1_checkpoints/`, flip
`TRAIN_CHECKPOINTS = True` below to train them in-session via
[`make_checkpoints.py`](../make_checkpoints.py). Full paper-grade training
(`TRAIN_MODE='shakespeare'`) is ~6 h on one GPU; `TRAIN_MODE='smoke'` runs in
minutes but is **not** paper-grade (pipeline check only). This cell is a no-op
when `TRAIN_CHECKPOINTS = False` (the default).

In [ ]:
# ===== Optional: (re)create the local checkpoints from scratch =====
# OFF by default. When True, trains any MISSING checkpoint in-session via
# make_checkpoints.py, writing it under <repo>/checkpoints/ with the exact
# filename the extraction stages expect, then re-enables that architecture's
# extraction arm (the config cell disables arms whose checkpoint is absent).
#   TRAIN_MODE='shakespeare' -> paper-grade (4000 steps each, ~6 h on one GPU)
#   TRAIN_MODE='smoke'       -> fast (300 steps each, minutes) but NOT paper-grade
TRAIN_CHECKPOINTS = False
TRAIN_MODE        = 'shakespeare'   # or 'smoke'

if TRAIN_CHECKPOINTS:
    need = []
    if not SPLM_CKPT.exists():    need.append('splm')
    if not MATCHED_CKPT.exists(): need.append('matched')
    if not need:
        print('[train] both checkpoints already present \u2014 nothing to do.', flush=True)
    else:
        which = 'all' if len(need) == 2 else need[0]
        print(f'[train] TRAIN_MODE={TRAIN_MODE!r}; recreating: {need}', flush=True)
        stream_subprocess(
            [sys.executable, 'make_checkpoints.py',
             '--mode', TRAIN_MODE, '--seed', str(SEED), '--which', which],
            cwd=CONS_DIR, label='make-checkpoints',
        )
    # Re-point to the freshly trained checkpoints under <repo>/checkpoints/ and
    # re-enable their extraction arms.
    _ck      = REPO_ROOT / 'checkpoints'
    _splm    = _ck / f'splm_em_ln_leakfree_gamma0p10_seed{SEED}_ckpt_latest.pt'
    _matched = _ck / f'matched_baseline_shakespeare_seed{SEED}_ckpt_latest.pt'
    if _splm.exists():
        SPLM_CKPT = _splm;       RUN_SPLM = True
    if _matched.exists():
        MATCHED_CKPT = _matched; RUN_MATCHED = True
    if IN_COLAB:   # persist to Drive so future sessions skip retraining
        _dst = Path('/content/drive/MyDrive') / 'paper_tmlr_1_checkpoints'
        _dst.mkdir(parents=True, exist_ok=True)
        for _p in (_splm, _matched):
            if _p.exists():
                shutil.copy2(_p, _dst / _p.name)
                print(f'[train] persisted -> {_dst / _p.name}', flush=True)
    print(f'[train] SPLM_CKPT    = {SPLM_CKPT}  (exists={SPLM_CKPT.exists()}, RUN_SPLM={RUN_SPLM})', flush=True)
    print(f'[train] MATCHED_CKPT = {MATCHED_CKPT}  (exists={MATCHED_CKPT.exists()}, RUN_MATCHED={RUN_MATCHED})', flush=True)
else:
    print('[train] TRAIN_CHECKPOINTS=False \u2014 using pre-existing checkpoints '
          '(arms whose checkpoint is absent stay skipped).', flush=True)

## 3. Stage 1a — Extract pretrained GPT-2 small trajectories

In [ ]:
PER_ARCH = {}

if RUN_GPT2:
    gpt2_pkl = CONS_DIR / 'results' / 'gpt2_baseline.trajectories.pkl'
    stream_subprocess(
        [sys.executable, 'extract_gpt2_baseline.py',
         '--model', 'gpt2',
         '--out',   str(gpt2_pkl),
         '--n_test_per_domain', str(N_TEST_PER_DOMAIN),
         '--max_len', str(MAX_LEN),
         '--seed',    str(SEED)],
        cwd=CONS_DIR, label='extract-gpt2',
    )
    assert gpt2_pkl.exists(), f'missing: {gpt2_pkl}'
    PER_ARCH['gpt2_small_pretrained'] = {
        'tag':        'gpt2_baseline',
        'arch_label': 'gpt2_small_pretrained',
        'traj':       gpt2_pkl,
        'ckpt':       None,
    }
    print(f'GPT-2 trajectories: {gpt2_pkl}  size={gpt2_pkl.stat().st_size / 1e6:.1f} MB', flush=True)

## 4. Stage 1b — Extract MatchedGPT 8M trajectories (gated on local checkpoint)

In [ ]:
if RUN_MATCHED:
    matched_pkl = CONS_DIR / 'results' / 'matched_baseline.trajectories.pkl'
    stream_subprocess(
        [sys.executable, 'extract_matched_baseline.py',
         '--ckpt',  str(MATCHED_CKPT),
         '--out',   str(matched_pkl),
         '--n_test_per_domain', str(N_TEST_PER_DOMAIN),
         '--max_len', str(MAX_LEN),
         '--seed',    str(SEED)],
        cwd=CONS_DIR, label='extract-matched',
    )
    assert matched_pkl.exists(), f'missing: {matched_pkl}'
    PER_ARCH['matched_attention_8M'] = {
        'tag':        'matched_baseline',
        'arch_label': 'matched_attention_8M',
        'traj':       matched_pkl,
        'ckpt':       MATCHED_CKPT,
    }
    print(f'MatchedGPT trajectories: {matched_pkl}  size={matched_pkl.stat().st_size / 1e6:.1f} MB', flush=True)
else:
    print('SKIPPED — matched-attention checkpoint not available.', flush=True)

## 5. Stage 1c — Extract SPLM leak-free positive-control trajectories (gated on local checkpoint)

In [ ]:
if RUN_SPLM:
    splm_pkl = CONS_DIR / 'results' / 'splm_leakfree_gamma0p10_seed0.trajectories.pkl'
    stream_subprocess(
        [sys.executable, 'trajectory_extraction.py',
         '--ckpt',  str(SPLM_CKPT),
         '--out',   str(splm_pkl),
         '--n_test_per_domain', str(N_TEST_PER_DOMAIN),
         '--max_len', str(MAX_LEN),
         '--seed',    str(SEED)],
        cwd=CONS_DIR, label='extract-splm',
    )
    assert splm_pkl.exists(), f'missing: {splm_pkl}'
    PER_ARCH['splm_leakfree_gamma0p10'] = {
        'tag':        'splm_leakfree_gamma0p10_seed0',
        'arch_label': 'splm_leakfree_gamma0p10',
        'traj':       splm_pkl,
        'ckpt':       SPLM_CKPT,
    }
    print(f'SPLM trajectories: {splm_pkl}  size={splm_pkl.stat().st_size / 1e6:.1f} MB', flush=True)
else:
    print('SKIPPED — SPLM checkpoint not available.', flush=True)

## 6. Stage 2 — `shared_potential_fit.py` per architecture

Fits the strict shared-$V_\psi$ test on each trajectory, emitting one JSON per architecture. These three JSONs feed the §8 three-way separator aggregator in Stage 6.

In [ ]:
PER_ARCH_SHARED_V_JSON = {}
for arch_label, info in PER_ARCH.items():
    out_json = RESULTS_ROOT / f'sharedV_{arch_label}.json'
    stream_subprocess(
        [sys.executable, 'shared_potential_fit.py',
         '--traj',   str(info['traj']),
         '--tag',    info['tag'],
         '--hidden', str(SHARED_V_HIDDEN),
         '--depth',  str(SHARED_V_DEPTH),
         '--steps',  str(SHARED_V_STEPS),
         '--batch',  str(SHARED_V_BATCH),
         '--lr',     str(SHARED_V_LR),
         '--device', device,
         '--seed',   str(SEED),
         '--architecture-label', arch_label,
         '--emit-json', str(out_json)],
        cwd=CONS_DIR, label=f'shared_v[{arch_label}]',
    )
    assert out_json.exists(), f'missing: {out_json}'
    PER_ARCH_SHARED_V_JSON[arch_label] = out_json
    print(f'[{arch_label}] sharedV JSON: {out_json}', flush=True)

## 7. Stage 3 — `splm_oracle_fit.py` (SPLM only, gated on the SPLM checkpoint)

In [ ]:
R2_ORACLE_JSON = None
if RUN_SPLM:
    R2_ORACLE_JSON = RESULTS_ROOT / 'R2_oracle.json'
    stream_subprocess(
        [sys.executable, 'splm_oracle_fit.py',
         '--ckpt',   str(SPLM_CKPT),
         '--tag',    'splm_leakfree_gamma0p10',
         '--device', device,
         '--seed',   str(SEED),
         '--emit-json', str(R2_ORACLE_JSON)],
        cwd=CONS_DIR, label='splm_oracle',
    )
    assert R2_ORACLE_JSON.exists(), f'missing: {R2_ORACLE_JSON}'
    print(f'splm oracle JSON: {R2_ORACLE_JSON}', flush=True)
else:
    print('SKIPPED — splm_oracle_fit requires the SPLM checkpoint.', flush=True)

## 8. Stage 4 — `sharedV_capacity_sweep.py` (GPT-2 only)

Six capacity cells × 3000 AdamW steps each. The script is the slow one in this notebook (~30 min on H100). It writes the per-cell `.npz` artefacts into `notebooks/conservative_arch/results/` (in-tree) and the aggregated JSON to `RESULTS_ROOT/R2_capacity_sweep.json`.

In [ ]:
R2_CAPACITY_SWEEP_JSON = None
if RUN_GPT2:
    R2_CAPACITY_SWEEP_JSON = RESULTS_ROOT / 'R2_capacity_sweep.json'
    stream_subprocess(
        [sys.executable, 'sharedV_capacity_sweep.py',
         '--emit-json', str(R2_CAPACITY_SWEEP_JSON)],
        cwd=CONS_DIR, label='capacity_sweep',
    )
    assert R2_CAPACITY_SWEEP_JSON.exists(), f'missing: {R2_CAPACITY_SWEEP_JSON}'
    print(f'capacity-sweep JSON: {R2_CAPACITY_SWEEP_JSON}', flush=True)
else:
    print('SKIPPED — capacity sweep requires the GPT-2 trajectories.', flush=True)

## 9. Stage 5 — `token_direction_fit.py` per architecture

Three runs, one per architecture; each emits its own `R2_token_direction_<arch>.json`.

In [ ]:
PER_ARCH_TOKEN_DIR_JSON = {}
for arch_label, info in PER_ARCH.items():
    out_json = RESULTS_ROOT / f'R2_token_direction_{arch_label}.json'
    stream_subprocess(
        [sys.executable, 'token_direction_fit.py',
         '--traj',   str(info['traj']),
         '--tag',    info['tag'],
         '--hidden', str(TOKEN_DIR_HIDDEN),
         '--depth',  str(TOKEN_DIR_DEPTH),
         '--steps',  str(TOKEN_DIR_STEPS),
         '--batch',  str(TOKEN_DIR_BATCH),
         '--lr',     str(TOKEN_DIR_LR),
         '--pca_k',  str(TOKEN_DIR_PCA_K),
         '--ridge',  str(TOKEN_DIR_RIDGE),
         '--t_skip', str(TOKEN_DIR_T_SKIP),
         '--device', device,
         '--seed',   str(SEED),
         '--architecture-label', arch_label,
         '--emit-json', str(out_json)],
        cwd=CONS_DIR, label=f'token_dir[{arch_label}]',
    )
    assert out_json.exists(), f'missing: {out_json}'
    PER_ARCH_TOKEN_DIR_JSON[arch_label] = out_json
    print(f'[{arch_label}] token-dir JSON: {out_json}', flush=True)

## 10. Stage 6 — Aggregate the §8 three-way separator JSON

Requires all three per-architecture `shared_potential_fit` JSONs to be present. If any architecture was skipped, this stage is skipped and you'll need to fill in the missing input before the aggregator can run.

In [ ]:
R2_THREE_WAY_JSON = RESULTS_ROOT / 'R2_three_way_separator.json'
needed = ['splm_leakfree_gamma0p10', 'matched_attention_8M', 'gpt2_small_pretrained']
missing = [a for a in needed if a not in PER_ARCH_SHARED_V_JSON]
if missing:
    print(f'SKIPPED — three-way aggregator needs all three architectures; missing: {missing}', flush=True)
else:
    stream_subprocess(
        [sys.executable, str(SCRIPTS_DIR / 'build_r2_three_way_separator.py'),
         '--splm',    str(PER_ARCH_SHARED_V_JSON['splm_leakfree_gamma0p10']),
         '--matched', str(PER_ARCH_SHARED_V_JSON['matched_attention_8M']),
         '--gpt2',    str(PER_ARCH_SHARED_V_JSON['gpt2_small_pretrained']),
         '--out',     str(R2_THREE_WAY_JSON)],
        cwd=SCRIPTS_DIR, label='three_way_aggregate',
    )
    assert R2_THREE_WAY_JSON.exists(), f'missing: {R2_THREE_WAY_JSON}'
    print(f'three-way JSON: {R2_THREE_WAY_JSON}', flush=True)

## 11. Stage 7 — Display the produced JSONs inline

Pretty-prints the headline blocks of every JSON that was produced, so you can sanity-check the numbers against the paper's cited values before downloading.

In [ ]:
import json
from pathlib import Path

produced = []
if R2_THREE_WAY_JSON.exists():
    produced.append(('R2_three_way_separator', R2_THREE_WAY_JSON))
if R2_ORACLE_JSON is not None and R2_ORACLE_JSON.exists():
    produced.append(('R2_oracle', R2_ORACLE_JSON))
if R2_CAPACITY_SWEEP_JSON is not None and R2_CAPACITY_SWEEP_JSON.exists():
    produced.append(('R2_capacity_sweep', R2_CAPACITY_SWEEP_JSON))
for arch_label, p in PER_ARCH_TOKEN_DIR_JSON.items():
    produced.append((f'R2_token_direction[{arch_label}]', p))
for arch_label, p in PER_ARCH_SHARED_V_JSON.items():
    produced.append((f'sharedV[{arch_label}] (input to three-way aggregator)', p))

print(f'\n=== Headline summary ({len(produced)} JSONs produced) ===\n', flush=True)
for label, path in produced:
    print(f'\n--- {label} ---', flush=True)
    print(f'    path: {path}', flush=True)
    try:
        with path.open('r') as f:
            obj = json.load(f)
        headline_keys = [k for k in obj.keys() if k.startswith('headline')
                         or k in ('verdict', 'section')]
        for k in headline_keys:
            print(f'    {k}:')
            print(json.dumps(obj[k], indent=4))
    except Exception as e:
        print(f'    (failed to load: {e})', flush=True)

print('\n=== All produced JSONs are saved under RESULTS_ROOT ===', flush=True)
print(f'    RESULTS_ROOT = {RESULTS_ROOT}', flush=True)
print('Download these and commit them under `results/` in the standalone repo.', flush=True)